# 01: Datasets and Preprocessing

This notebook demonstrates the data exploration and preprocessing capabilities of the  framework, matching the visualizations found in the interactive dashboard.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from fishy.data.module import create_data_module


## 1. Load Dataset and Basic Metrics

We'll start by loading the  dataset using the .

In [ ]:
dataset_name = "species"
dm = create_data_module(dataset_name)
dm.setup()
df = dm.get_filtered_dataframe()

print(f"Total Samples: {len(df)}")
print(f"Feature Count (m/z bins): {dm.get_input_dim()}")

## 2. Class Distribution

Understanding the balance of classes is crucial for model evaluation.

In [ ]:
dist = df["Class Name"].value_counts()
px.bar(x=dist.index, y=dist.values, labels={'x': 'Class', 'y': 'Count'}, 
       title="Class Distribution", template="plotly_white")

## 3. Mean Spectral Signatures

Visualizing the average spectrum per class with standard deviation shading to identify characteristic peaks.

In [ ]:
class_names = dm.get_class_names()
X_all, y_all = dm.get_numpy_data(labels_as_indices=True)
# Extract m/z values from column names (dropping non-numeric metadata columns)
label_col = "Class Name"
feature_cols = [c for c in df.columns if c not in [label_col, "m/z"]]
mz_axis = np.array([float(c) for c in feature_cols])

avg_fig = go.Figure()
for idx, name in enumerate(class_names):
    mask = (y_all == idx)
    if np.any(mask):
        m, s = X_all[mask].mean(axis=0), X_all[mask].std(axis=0)
        avg_fig.add_trace(go.Scatter(x=mz_axis, y=m, name=name, mode="lines"))
        avg_fig.add_trace(go.Scatter(x=np.concatenate([mz_axis, mz_axis[::-1]]), 
                                     y=np.concatenate([m+s, (m-s)[::-1]]), 
                                     fill="toself", fillcolor="rgba(0,100,80,0.1)", 
                                     line=dict(color="rgba(255,255,255,0)"), 
                                     hoverinfo="skip", showlegend=False))

avg_fig.update_layout(template="plotly_white", xaxis_title="m/z", yaxis_title="Intensity", height=500)
avg_fig.show()

## 4. Interactive Spectrum Viewer

Overlaying individual samples to see the raw variance within classes.

In [ ]:
sample_df = df.sample(min(10, len(df)))
melted = sample_df[[label_col]+feature_cols].melt(id_vars=label_col, var_name="m/z", value_name="Intensity")
px.line(melted, x="m/z", y="Intensity", color=label_col, 
        title="Overlay of Individual Samples", template="plotly_white")

## 5. Dimensionality Reduction

Using PCA and t-SNE to visualize class clusters in high-dimensional space.

In [ ]:
# PCA
pca_obj = PCA(n_components=10)
X_pca = pca_obj.fit_transform(X_all)
pca_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(10)])
pca_df["Class"] = [class_names[i] for i in y_all]

fig_pca = px.scatter(pca_df, x="PC1", y="PC2", color="Class", 
                     title="PCA: First Two Principal Components", template="plotly_white")
fig_pca.show()

# PCA Information Retention
px.area(x=range(1, 11), y=np.cumsum(pca_obj.explained_variance_ratio_), 
        labels={'x': 'Components', 'y': 'Cumulative Explained Variance'}, 
        title="PCA Information Retention", template="plotly_white").show()

In [ ]:
# t-SNE
tsne = TSNE(n_components=2, perplexity=min(30, len(X_all)-1), random_state=42)
X_tsne = tsne.fit_transform(X_all)
tsne_df = pd.DataFrame(X_tsne, columns=["TSNE1", "TSNE2"])
tsne_df["Class"] = [class_names[i] for i in y_all]

px.scatter(tsne_df, x="TSNE1", y="TSNE2", color="Class", 
           title="t-SNE Visualization", template="plotly_white").show()

## 6. Intensity Distribution

Checking for global intensity differences between classes using violin plots.

In [ ]:
avg_int = pd.DataFrame({"Avg Intensity": X_all.mean(axis=1), "Class": [class_names[i] for i in y_all]})
px.violin(avg_int, x="Class", y="Avg Intensity", color="Class", 
          box=True, points="all", title="Global Intensity Distribution", template="plotly_white")